In [2]:
from itertools import product
import numpy as np
import random
import os
import shutil

In [ ]:
# TSGym_xmark_sampling_norm_decomp_ci_inputdim_network_attn_featureattn_encoderonly_frozen_hp_seqlen_dmodel_el_epoch_lossfn_lr_strategy
SOTA_MODELS_NonTransformers = [
    "TSGym_False_False_RevIn_MA_True_series-encoding_MLP_null_self-attention_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
    "TSGym_False_False_Stat_None_False_inverted-encoding_MLP_null_self-attention_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
    "TSGym_False_False_Stat_None_CI_series-patching_GRU_null_null_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
    "TSGym_False_True_RevIN_MA_False_series-encoding_MLP_null_null_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
    "TSGym_False_False_None_MA_True_series-encoding_MLP_null_null_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
    "TSGym_False_False_Stat_DFT_False_series-encoding_MLP_null_null_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
    "TSGym_False_False_Stat_DFT_True_series-patching_MLP_null_null_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
    "TSGym_False_True_RevIN_None_CI_series-encoding_MLP_null_null_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False"
    "TSGym_False_False_RevIN_None_CD_ortho-encoding_MLP_NormLin_null_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False"
]


SOTA_MODELS_Transformers = [
    "TSGym_True_False_None_None_CI_series-patching_Transformer_self-attention_null_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
    "TSGym_False_False_Stat_None_CI_series-patching_Transformer_self-attention_null_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
    "TSGym_True_False_None_None_CI_series-encoding_Transformer_self-attention_null_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
    "TSGym_True_False_None_DFT_False_series-encoding_Transformer_destationary-attention_null_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
    "TSGym_True_None_MoEMA_False_series-encoding_Transformer_frequency-enhanced-attention_null_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
    "TSGym_True_True_None_None_False_series-encoding_Transformer_self-attention_null_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
    "TSGym_False_False_None_None_False_series-encoding_Transformer_sparse-attention_null_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
    "TSGym_True_False_None_None_False_series-encoding_Transformer_auto-correlation_null_True_False_False_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
]

SOTA_MODELS_LLM = [
    "TSGym_True_False_Stat_None_CI_series-patching_LLM-GPT4TS_self-attention_null_True_False_True_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False",
    "TSGym_True_False_Stat_None_CI_series-patching_LLM-TimeLLM_self-attention_null_True_False_True_HP_seqlen_dmodel_elayers_30_MSE_lr_lrs_False"
]

SOTA_MODELS_TSFM = [

]

In [ ]:
def wrong_setting(series_sampling, series_norm,series_decomp, channel_independent, input_embed, network_architecture, attn, feature_attn, gym_frozen, gym_rag):
    if series_sampling and input_embed == 'inverted-encoding':
        return True
    if channel_independent and input_embed == 'inverted-encoding':
        return True
    if not channel_independent and input_embed == 'series-patching':
        return True
    if network_architecture == 'Transformer' and attn == 'null':
        return True
    # if network_architecture in ['GRU','MLP'] and attn != 'null':
    #     return True
    if attn == 'destationary-attention' and series_norm != 'Stat':
        return True
    if attn == 'destationary-attention' and input_embed != 'series-encoding':
        return True
    if channel_independent and feature_attn != 'null':
        return True
    if gym_frozen and network_architecture not in ['LLM-GPT4TS', 'LLM-TimeLLM','TSFM-Timer', 'TSFM-Moment']:
        return True
    if network_architecture in ['LLM-GPT4TS', 'LLM-TimeLLM','TSFM-Timer', 'TSFM-Moment'] and attn != 'self-attention':
        return True
    if network_architecture in ['GRU','MLP'] and input_embed == 'inverted-encoding':
        return True
    if input_embed == 'inverted-encoding' and attn not in ['self-attention', 'sparse-attention', 'null']:
        return True
    if gym_rag and series_sampling:
        return True
    if input_embed == 'ortho-encoding' and channel_independent:
        return True
    if input_embed == 'ortho-encoding' and network_architecture == ['GRU']:
        return True
    if input_embed == 'ortho-encoding' and series_decomp!="None":
        return True
    if attn == 'NormLin' and network_architecture != 'MLP':
        return True
    if attn == 'GRU' and  network_architecture != 'GRU':
        return True
    return False

In [ ]:
def generate_components(gym_type):
    if gym_type == 'Transformer':
        model_names = ['_'.join(list(_)) for _ in list(product(['TSGym'],
                                                            ['False', 'True'], # gym_x_mark
                                                            ['False', 'True'], # gym_series_sampling
                                                            ['None', 'Stat', 'RevIN', 'DishTS'], # gym_series_norm
                                                            ['None', 'MA', 'MoEMA', 'DFT'], # gym_series_decomp
                                                            ['False', 'True'], # gym_channel_independent
                                                            ['inverted-encoding', 'series-encoding', 'series-patching'], # gym_input_embed
                                                            ['Transformer'], # gym_network_architecture
                                                            ['self-attention', 'auto-correlation', 'sparse-attention', 'frequency-enhanced-attention', 'destationary-attention'], # gym_attn
                                                            ['null', 'self-attention', 'sparse-attention'], # gym_feature_attn， null
                                                            ['True'], # gym_encoder_only
                                                            ['False'], # gym_frozen
                                                            ['HP'],
                                                            ['48', '96', '192', '512'], # sequence length 
                                                            ['64-256', '256-1024'], # d_model, d_ff
                                                            ['2', '3'], # encoder layers
                                                            ['30'], # training epochs
                                                            ['MSE', 'MAE', 'HUBER', 'MAPE', 'DBLoss', 'PSLoss', 'FreDFLoss'], # loss functions
                                                            ['0.0001'], # learning rate
                                                            ['null','cosine'], # learning rate strategy
                                                            ['True','False'] # gym_rag
                                                            ))]
    elif gym_type == 'LLM':
        model_names = ['_'.join(list(_)) for _ in list(product(['TSGym'],
                                                            ['False', 'True'], # gym_x_mark
                                                            ['False'], # gym_series_sampling
                                                            ['None', 'Stat', 'RevIN', 'DishTS'], # gym_series_norm
                                                            ['None', 'MA', 'MoEMA', 'DFT'], # gym_series_decomp
                                                            ['True'], # gym_channel_independent
                                                            ['series-patching'], # gym_input_embed
                                                            ['LLM-GPT4TS', 'LLM-TimeLLM'], # gym_network_architecture
                                                            ['self-attention'], # gym_attn
                                                            ['null'], # gym_feature_attn
                                                            ['True'], # gym_encoder_only
                                                            ['False', 'True'], # gym_frozen
                                                            ['HP'],
                                                            ['48', '96', '192', '512'], # sequence length 
                                                            ['64-256', '256-1024'], # d_model, d_ff
                                                            ['6'], # encoder layers
                                                            ['30'], # training epochs
                                                            ['MSE', 'MAE', 'HUBER', 'MAPE', 'DBLoss', 'PSLoss', 'FreDFLoss'], # loss functions
                                                            ['0.0001'], # learning rate
                                                            ['null','cosine'], # learning rate strategy
                                                            ['True','False'] # gym_rag
                                                            ))]
    elif gym_type == 'TSFM':
        model_names = ['_'.join(list(_)) for _ in list(product(['TSGym'],
                                                            ['False', 'True'], # gym_x_mark
                                                            ['False'], # gym_series_sampling
                                                            ['None', 'Stat', 'RevIN', 'DishTS'], # gym_series_norm
                                                            ['None', 'MA', 'MoEMA', 'DFT'], # gym_series_decomp
                                                            ['True'], # gym_channel_independent
                                                            ['series-patching'], # gym_input_embed
                                                            ['TSFM-Timer', 'TSFM-Moment', 'TSFM-TimeMoE', 'TSFM-Chronos', 'TSFM-TimerXL'], # gym_network_architecture
                                                            ['self-attention'], # gym_attn
                                                            ['null'], # gym_feature_attn
                                                            ['True'], # gym_encoder_only
                                                            ['False', 'True'], # gym_frozen
                                                            ['HP'],
                                                            ['48', '96', '192', '512'], # sequence length 
                                                            ['64-256', '256-1024'], # d_model, d_ff
                                                            ['6'], # encoder layers
                                                            ['30'], # training epochs
                                                            ['MSE', 'MAE', 'HUBER', 'MAPE', 'DBLoss', 'PSLoss', 'FreDFLoss'], # loss functions
                                                            ['0.001'], # learning rate
                                                            ['null','cosine'], # learning rate strategy
                                                            ['True','False'] # gym_rag
                                                            ))]
    else:
        model_names = ['_'.join(list(_)) for _ in list(product(['TSGym'],
                                                        ['False', 'True'], # gym_x_mark
                                                        ['False', 'True'], # gym_series_sampling
                                                        ['None', 'Stat', 'RevIN', 'DishTS'], # gym_series_norm
                                                        ['None', 'MA', 'MoEMA', 'DFT'], # gym_series_decomp
                                                        ['False', 'True'], # gym_channel_independent
                                                        ['inverted-encoding', 'series-encoding', 'series-patching'], # gym_input_embed
                                                        ['MLP', 'GRU'], # gym_network_architecture
                                                        ['GRU','DNN','NormLin','xLSTM'], # gym_attn
                                                        ['null', 'self-attention', 'sparse-attention'], # gym_feature_attn, null
                                                        ['True'], # gym_encoder_only
                                                        ['False'], # gym_frozen
                                                        ['HP'],
                                                        ['48', '96', '192', '512'], # sequence length ['48', '96', '192']
                                                        ['64-256', '256-1024'], # d_model, d_ff
                                                        ['2', '3'], # encoder layers
                                                        ['30'], # training epochs
                                                        ['MSE', 'MAE', 'HUBER', 'MAPE', 'DBLoss', 'PSLoss', 'FreDFLoss'], # loss functions
                                                        ['0.001'], # learning rate
                                                        ['null','cosine'], # learning rate strategy
                                                        ['True','False'] # gym_rag
                                                        ))]
    print(f"Generated {len(model_names)} model names for gym type '{gym_type}'.")
    valid_model_names = []
    for name in model_names:
        components = name.split('_')
        series_sampling = components[2] == 'True'
        series_norm = components[3]
        series_decomp = components[4]
        channel_independent = components[5] == 'True'
        input_embed = components[6]
        network_architecture = components[7]
        attn = components[8]
        feature_attn = components[9]
        gym_frozen = components[11] == 'True'
        gym_rag = components[-1] == 'True'
        if not wrong_setting(series_sampling, series_norm, series_decomp, channel_independent, input_embed, network_architecture, attn, feature_attn, gym_frozen, gym_rag):
            valid_model_names.append(name)
    print(f"Filtered down to {len(valid_model_names)} valid model names for gym type '{gym_type}'.")
    return valid_model_names

In [4]:
from collections import defaultdict
import random

def grouped_sampling(valid_model_names, target_sample_size, comparison_indices):
    """
    进行分组采样，确保生成的样本可以用于消融实验。
    
    :param valid_model_names: 所有合法的模型名称列表
    :param target_sample_size: 目标采样的总数量 (如 5000)
    :param comparison_indices: 你希望变化的维度的索引列表 (例如 attn 在 split('_') 后的索引是 8)
                               这些维度在生成 Group Key 时会被忽略。
    """
    groups = defaultdict(list)
    
    # 1. 遍历所有模型，根据“非对比维度”进行分组
    for name in valid_model_names:
        components = name.split('_')
        
        # 创建指纹 (Signature): 只有当索引不在 comparison_indices 中时才加入 Key
        # 这样，除了我们想对比的变量外，其他完全一样的模型会拥有相同的 Key
        signature_parts = [
            comp for i, comp in enumerate(components) 
            if i not in comparison_indices
        ]
        signature = '_'.join(signature_parts)
        groups[signature].append(name)
        
    # 2. 过滤掉那些只有一个孤立模型的组 (因为没有对照组，无法进行比较)
    #    或者你可以保留它们，取决于你是否只关心成对比较
    comparable_groups = [g for g in groups.values() if len(g) > 1]
    
    print(f"Found {len(comparable_groups)} groups suited for ablation studies out of {len(groups)} total signatures.")
    
    # 3. 开始采样组，直到达到目标数量
    sampled_models = []
    
    # 打乱组的顺序
    random.shuffle(comparable_groups)
    
    for group in comparable_groups:
        if len(sampled_models) >= target_sample_size:
            break
        
        # 将整组加入 (这样你就拥有了完美的对照组)
        # 如果组太大，也可以只随机取组内的 n 个，但在你的空间里通常不大
        sampled_models.extend(group)
        
    # 如果因为组大小原因稍微超过了 target_sample_size，通常是可以接受的
    # 如果必须严格限制数量，可以截断
    return sampled_models

In [5]:
from collections import defaultdict

def calculate_component_proportions(strings):
    # 分割字符串并统计每个组件出现的次数
    component_counts = [defaultdict(int) for _ in range(len(strings[0].split('_')))]
    
    for s in strings:
        components = s.split('_')
        for i, comp in enumerate(components[1:]):  # 跳过第一个组件 'TSGym'
            component_counts[i][comp] += 1
    
    # 计算每个组件的占比
    proportions = []
    for counts in component_counts:
        total_count = sum(counts.values())
        proportions.append({key: round(count / total_count, 2) for key, count in counts.items()})
    
    return proportions

In [ ]:
from tqdm import tqdm
for setting_idx, gym_type in enumerate(['Transformer', 'non_Transformer', 'LLM', 'TSFM']):
    random.seed(42)
    model_names = generate_components(gym_type)
    gym_xmark = grouped_sampling(model_names, 200, [1])
    gym_seriesmixing = grouped_sampling(model_names, 200, [2])
    gym_seriesnorm = grouped_sampling(model_names, 400, [3])
    gym_seriesdecomp = grouped_sampling(model_names, 400, [4])
    gym_ci = grouped_sampling(model_names, 200, [5])
    gym_inputdim = grouped_sampling(model_names, 300, [6])
    gym_network = grouped_sampling(model_names, 200, [7])
    gym_attn = grouped_sampling(model_names, 500, [8])
    gym_featureattn = grouped_sampling(model_names, 300, [9])
    gym_lossfn = grouped_sampling(model_names, 700, [16])
    gym_lr = grouped_sampling(model_names, 200, [18])
    model_names = gym_xmark+gym_seriesmixing+gym_seriesnorm+gym_seriesdecomp+gym_ci+gym_inputdim+gym_network+gym_attn+gym_featureattn+gym_lossfn+gym_lr
    print(f"gen scripts:{len(model_names)}")
    print(f"Calculating component proportions for gym type '{gym_type}'...")
    print(calculate_component_proportions(model_names))
    # 给每个setting设置一个编号,long term forecasting用1起始
    model_names = [m.replace("TSGym", f"TSGym1{setting_idx}{str(i).zfill(4)}") for i,m in enumerate(model_names)]

    for dataset in ['ETTh1','ECL', 'ETTh1', 'ETTh2', 'ETTm1', 'ETTm2', 'Exchange', 'ILI', 'Traffic', 'Weather','NYSE','NASDAQ']:#
    # for dataset in ['covid-19', 'fred-md']:
        # 模板文件路径
        if 'ETT' in dataset:
            template_path = f'../scripts/long_term_forecast/{dataset}_script/TSGym_{dataset}.sh'
        else:
            template_path = f'../scripts/long_term_forecast/{dataset}_script/TSGym.sh'
            
        # 输出目录
        output_dir = f'../scripts/long_term_forecast/{dataset}_script/gym_{gym_type}'

        # 确保输出目录存在
        if os.path.exists(output_dir):
            print(f'delete current folder for dataset: {dataset}!')
            shutil.rmtree(output_dir)
        os.makedirs(output_dir, exist_ok=True)

        # 读取模板内容
        with open(template_path, 'r') as file:
            template_content = file.read()

        # 对于每个模型名称，生成一个 shell 脚本
        for model_name in tqdm(model_names):
            file_name = model_name

            HP = model_name[model_name.find('_HP')+4:]
            model_name = model_name[:model_name.find('_HP')]

            seq_len, dm_df, el, epochs, loss, lr, lr_strategy = HP.split('_')
            dm, df = dm_df.split('-')[0], dm_df.split('-')[1]

            # 替换模型名称
            script_content = template_content.replace('$model_name', model_name)
            script_content = script_content.replace(f'$seq_len', seq_len)
            script_content = script_content.replace(f'$d_model', dm)
            script_content = script_content.replace(f'$d_ff', df)
            script_content = script_content.replace(f'$e_layers', el)
            script_content = script_content.replace(f'$train_epochs', epochs)
            script_content = script_content.replace(f'$loss', loss)
            script_content = script_content.replace(f'$learning_rate', lr)
            script_content = script_content.replace(f'$lradj', lr_strategy)
            
            # 定义输出文件名
            output_file = os.path.join(output_dir, f'{file_name}.sh')
            
            # 写入新的 shell 脚本
            with open(output_file, 'w') as file:
                file.write(script_content)

            # print(f'Generated {output_file}')

Generated 1290240 model names for gym type 'Transformer'.
Filtered down to 344064 valid model names for gym type 'Transformer'.
Found 172032 groups suited for ablation studies out of 172032 total signatures.
Found 150528 groups suited for ablation studies out of 193536 total signatures.
Found 82432 groups suited for ablation studies out of 96768 total signatures.
Found 86016 groups suited for ablation studies out of 86016 total signatures.
Found 60928 groups suited for ablation studies out of 283136 total signatures.
Found 100352 groups suited for ablation studies out of 243712 total signatures.
Found 0 groups suited for ablation studies out of 344064 total signatures.
Found 93184 groups suited for ablation studies out of 93184 total signatures.
Found 75264 groups suited for ablation studies out of 193536 total signatures.
Found 0 groups suited for ablation studies out of 344064 total signatures.
Found 0 groups suited for ablation studies out of 344064 total signatures.


In [7]:
model_names

['TSGym_False_False_Stat_MoEMA_True_series-patching_Transformer_auto-correlation_null_True_False_HP_192_64-256_2_30_MSE_0.001_cosine',
 'TSGym_True_False_Stat_MoEMA_True_series-patching_Transformer_auto-correlation_null_True_False_HP_192_64-256_2_30_MSE_0.001_cosine',
 'TSGym_False_False_RevIN_MoEMA_False_series-encoding_Transformer_frequency-enhanced-attention_sparse-attention_True_False_HP_96_256-1024_3_30_PSLoss_0.001_cosine',
 'TSGym_True_False_RevIN_MoEMA_False_series-encoding_Transformer_frequency-enhanced-attention_sparse-attention_True_False_HP_96_256-1024_3_30_PSLoss_0.001_cosine',
 'TSGym_False_True_None_DFT_False_series-encoding_Transformer_self-attention_self-attention_True_False_HP_96_64-256_2_30_DBLoss_0.001_null',
 'TSGym_True_True_None_DFT_False_series-encoding_Transformer_self-attention_self-attention_True_False_HP_96_64-256_2_30_DBLoss_0.001_null',
 'TSGym_False_False_RevIN_MA_False_series-encoding_Transformer_self-attention_sparse-attention_True_False_HP_192_256-1024